# SynthID-Text Tournament Watermark Visualization

This notebook reproduces the `cli_watermark_synthid.py` workflow with rich visual output.
Tokens are colored by their **hit strength** — how many of the `m` watermarking layers
score the token as "green".

## Color Scheme (viridis colormap)

Tokens are colored using a **continuous viridis colormap** mapped to hit strength (0 → dark purple, m → bright yellow):

- 🟪 **Dark purple** — low hit strength (weak watermark signal)
- 🟦 **Blue/Cyan** — moderate hit strength
- 🟩 **Green** — high hit strength (strong watermark signal)
- 🟨 **Bright yellow** — maximum hit strength (very strong watermark signal)

The color scale is normalized across all three texts so colors are directly comparable.

In [1]:
import numpy as np
from colorama import Fore, Style, init as colorama_init
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from target_hash_gen.core import Model, g_score, load_tokenizer, DEFAULT_SEED, _tok
from target_hash_gen.greedy import GreedyGenerator
from target_hash_gen.watermark_synthid import TournamentWatermarkGenerator

# Enable colorama for terminals
colorama_init(autoreset=True)
from IPython.display import display, HTML

# Viridis colormap (0=dark purple, 0.5=cyan/green, 1=bright yellow)
viridis_colors = [(0.267004, 0.004874, 0.329415),
                  (0.282638, 0.140899, 0.457590),
                  (0.253935, 0.265254, 0.529983),
                  (0.163625, 0.471133, 0.558148),
                  (0.134160, 0.658636, 0.517649),
                  (0.477504, 0.821366, 0.318195),
                  (0.993248, 0.906157, 0.143936)]
viridis_cmap = LinearSegmentedColormap.from_list('viridis_custom', viridis_colors, N=256)

def viridis_hex(t: float) -> str:
    """Map a float in [0,1] to a hex color using viridis."""
    t = max(0.0, min(1.0, t))
    rgb = viridis_cmap(t)
    return f'#{int(rgb[0]*255):02x}{int(rgb[1]*255):02x}{int(rgb[2]*255):02x}'


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [2]:
def hit_strength(seed: str, token_id: int, m: int) -> int:
    """How many of the m watermarking functions score this token green (0..m)."""
    return sum(g_score(seed, token_id, str(layer)) for layer in range(m))


def ref_quantiles(ids: list[int], seed: str, m: int) -> tuple[float, float, float]:
    """p50/p75/p90 of hit_strength across a reference text (e.g. the `wm`
    output), used to derive dynamic coloring thresholds."""
    hits = [hit_strength(seed, int(t), m) for t in ids]
    return tuple(np.percentile(hits, [50, 75, 90]))  # type: ignore[return-value]


def colored_text_html(
    ids: list[int], tok, seed: str, m: int = 5, p50: float = 0.0, p75: float = 0.0, p90: float = 0.0
) -> str:
    """Decode tokens colored by hit_strength using continuous viridis colormap."""
    parts: list[str] = []
    for t in ids:
        text = tok.decode([t], skip_special_tokens=True)
        hits = hit_strength(seed, int(t), m)
        # Normalize hits to [0, 1] for viridis
        norm_hits = hits / m  # 0.0 = dark purple, 1.0 = bright yellow
        color = viridis_hex(norm_hits)
        # Escape HTML special chars
        text = text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
        parts.append(f'<span style="color:{color}; background-color:white;font-size:1.5em" title="hits={hits}/{m}">{text}</span>')
    return "".join(parts)


def split_starts(tokens: list[int], tok) -> list[int]:
    """Token indices (1-based) of sentence starts, skipping too-short tails."""
    return [i + 1 for i, t in enumerate(tokens) if tok.decode([t]) in (".", "!", "?") and len(tokens[i + 1 :]) >= 16]


def make_colorbar_html(m: int) -> str:
    """Generate an HTML colorbar showing the viridis scale mapped to hit strength."""
    # Create gradient bar with tick labels
    steps = 50
    gradient_parts = []
    for i in range(steps):
        t = i / (steps - 1)
        color = viridis_hex(t)
        gradient_parts.append(f'<div style="width:{100/steps}%;height:20px;background-color:{color};display:inline-block;"></div>')
    gradient = "".join(gradient_parts)
    
    # Tick marks
    ticks = []
    for h in range(m + 1):
        pct = h / m * 100
        ticks.append(f'<span style="position:absolute;left:{pct}%;bottom:-18px;transform:translateX(-50%);font-size:11px;">{h}</span>')
    ticks_html = "".join(ticks)
    
    return f"""
<div style="position:relative;width:100%;max-width:600px;margin:10px auto;">
    <div style="display:flex;align-items:center;">
        <span style="font-size:11px;margin-right:8px;">0</span>
        <div style="flex:1;position:relative;">
            {gradient}
            <div style="position:relative;height:20px;">{ticks_html}</div>
        </div>
        <span style="font-size:11px;margin-left:8px;">{m}</span>
    </div>
    <div style="text-align:center;font-size:11px;margin-top:4px;color:#666;">Hit Strength (0 = weak → {m} = strong)</div>
</div>
"""

In [3]:
# Parameters
PROMPT = "What is gravity?"
MAX_TOKENS = 200
TOP_K = 20
TOP_P = 0.95
LAYERS = 5
COMPETITORS = 2
SEED = "a seed"
WRONG_SEED = "negative key"

display(HTML(f"""
<h3>Configuration</h3>
<table>
<tr><th>Parameter</th><th>Value</th></tr>
<tr><td>Prompt</td><td>{PROMPT}</td></tr>
<tr><td>Seed</td><td>{SEED}</td></tr>
<tr><td>Wrong seed</td><td>{WRONG_SEED}</td></tr>
<tr><td>Layers (m)</td><td>{LAYERS}</td></tr>
<tr><td>Competitors (k)</td><td>{COMPETITORS}</td></tr>
<tr><td>Max tokens</td><td>{MAX_TOKENS}</td></tr>
<tr><td>Top-k</td><td>{TOP_K}</td></tr>
<tr><td>Top-p</td><td>{TOP_P}</td></tr>
</table>
"""))

Parameter,Value
Prompt,What is gravity?
Seed,a seed
Wrong seed,negative key
Layers (m),5
Competitors (k),2
Max tokens,200
Top-k,20
Top-p,0.95


In [4]:
# Build prompt
messages = [
    {
        "role": "user",
        "content": PROMPT,
    },
]
prompt = _tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
prompt_ids = _tok(prompt, add_special_tokens=False)["input_ids"]

print(f"Prompt tokens: {len(prompt_ids)}")
print(f"Prompt text: {prompt[:200]}...")

Prompt tokens: 13

Prompt text: <|startoftext|><|im_start|>user
What is gravity?<|im_end|>
<|im_start|>assistant
...

In [5]:
# Generate text with all three strategies
gen_wm = TournamentWatermarkGenerator(
    seed=SEED,
    m=LAYERS,
    k=COMPETITORS,
    top_k=TOP_K,
    top_p=TOP_P,
)
gen_neg = TournamentWatermarkGenerator(
    seed=WRONG_SEED,
    m=LAYERS,
    k=COMPETITORS,
    top_k=TOP_K,
    top_p=TOP_P,
)
gen_plain = GreedyGenerator(
    top_k=TOP_K,
    top_p=TOP_P,
)

wm = gen_wm.generate(prompt_ids, max_new_tokens=MAX_TOKENS)
neg = gen_neg.generate(prompt_ids, max_new_tokens=MAX_TOKENS)
plain = gen_plain.generate(prompt_ids, max_new_tokens=MAX_TOKENS)
wm, neg, plain = (ids[len(prompt_ids) :] for ids in (wm, neg, plain))

print(f"Generated tokens — watermarked: {len(wm)}, negative-seed: {len(neg)}, plain: {len(plain)}")

Generated tokens — watermarked: 94, negative-seed: 157, plain: 200

In [6]:
# Compute reference quantiles from watermarked output
p50, p75, p90 = ref_quantiles(wm, SEED, LAYERS)

display(HTML(f"""
<h3>🎨 Color Scale</h3>
<p>Viridis colormap mapped to hit strength (0 → {LAYERS}):</p>
{make_colorbar_html(LAYERS)}
<table>
<tr><th>Hit Strength</th><th>Viridis Color</th><th>Interpretation</th></tr>
<tr><td>0</td><td style="background-color:{viridis_hex(0)};padding:4px 12px;border-radius:4px;">{viridis_hex(0)}</td><td>Weak watermark signal</td></tr>
<tr><td>{LAYERS//2}</td><td style="background-color:{viridis_hex(0.5)};padding:4px 12px;border-radius:4px;">{viridis_hex(0.5)}</td><td>Moderate watermark signal</td></tr>
<tr><td>{LAYERS}</td><td style="background-color:{viridis_hex(1.0)};padding:4px 12px;border-radius:4px;">{viridis_hex(1.0)}</td><td>Strong watermark signal</td></tr>
</table>
"""))
display(HTML(f'<p><em>Reference quantiles (from watermarked text): p50={p50:.2f}, p75={p75:.2f}, p90={p90:.2f}</em></p>'))

Hit Strength,Viridis Color,Interpretation
0,#440154,Weak watermark signal
2,#29788e,Moderate watermark signal
5,#fde724,Strong watermark signal


In [7]:
# Display watermarked output
title = f"Watermarked output ({LAYERS} layers, {COMPETITORS} competitors/match)"
display(HTML(f'<h3>🟩 {title}</h3><p><em>Seed: {SEED}</em></p>'))
display(HTML(colored_text_html(wm, _tok, SEED, LAYERS)))

In [8]:
# Display negative-seed output
display(HTML(f'<h3>⬜ Negative-seed output</h3><p><em>Seed: {WRONG_SEED} (evaluated against {SEED})</em></p>'))
display(HTML(colored_text_html(neg, _tok, SEED, LAYERS)))

In [9]:
# Display plain output
display(HTML(f'<h3>⬜ Plain output (baseline)</h3><p><em>No watermark seed applied</em></p>'))
display(HTML(colored_text_html(plain, _tok, SEED, LAYERS)))

In [10]:
# Detection results
display(HTML('<h3>🔍 Detection Results</h3>'))
results = {
    "Watermarked": gen_wm.check_hash(wm, SEED),
    "Negative-seed": gen_wm.check_hash(neg, SEED),
    "Plain": gen_wm.check_hash(plain, SEED),
}
html_rows = "".join(f"<tr><td>{name}</td><td>{val}</td></tr>" for name, val in results.items())
display(HTML(f"""
<table border="1" cellpadding="6">
<tr><th>Text</th><th>Hash Check (seed='{SEED}')</th></tr>
{html_rows}
</table>
"""))

Text,Hash Check (seed='a seed')
Watermarked,3.2288592281010997
Negative-seed,0.892288262810312
Plain,1.581138830084191


In [11]:
# Hash across splits
display(HTML('<h3>📊 Hash Across Text Splits</h3>'))
for name, text in [("watermarked", wm), ("negative-seed", neg), ("plain", plain)]:
    display(HTML(f'<h4>Splits of {name} text</h4>'))
    html_rows = ""
    for st in split_starts(text, _tok):
        span = text[st:]
        val = gen_wm.check_hash(span, SEED)
        html_rows += f"<tr><td>{st}</td><td>{val}</td></tr>"
    display(HTML(f"""
<table border="1" cellpadding="6">
<tr><th>From token</th><th>Hash (seed='{SEED}')</th></tr>
{html_rows}
</table>
"""))

From token,Hash (seed='a seed')
22,3.162277660168381
53,2.7238781535013543


From token,Hash (seed='a seed')
17,0.9827076298239908
45,1.098700531147073
85,0.527046276694728
119,0.7254762501100109


From token,Hash (seed='a seed')
19,1.6288151135206093
49,1.4193553242313361
80,1.22474487139159
128,1.6865480854231338
156,0.4045199174779462
180,-0.20000000000000018


In [12]:
# Hit strength distribution
display(HTML('<h3>📈 Hit Strength Distribution</h3>'))
for name, text in [("watermarked", wm), ("negative-seed", neg), ("plain", plain)]:
    hits = [hit_strength(SEED, t, LAYERS) for t in text]
    if hits:
        mean_h = np.mean(hits)
        std_h = np.std(hits)
        counts = [hits.count(i) for i in range(LAYERS + 1)]
        display(HTML(f'<h4>{name} (mean={mean_h:.2f}, std={std_h:.2f})</h4>'))
        max_count = max(counts) if counts else 1
        bar_width = 40
        bars = ""
        for i, c in enumerate(counts):
            bar_len = int(c / max_count * bar_width) if max_count else 0
            bar = '█' * bar_len
            color = viridis_hex(i / LAYERS)
            bars += f"<tr><td>hits={i}</td><td style='color:{color};font-weight:bold'>{bar} ({c})</td></tr>"
        display(HTML(f"""
<table border="1" cellpadding="6">
<tr><th>Hits</th><th>Distribution</th></tr>
{bars}
</table>
"""))

Hits,Distribution
hits=0,(0)
hits=1,█████████ (8)
hits=2,████████████████████████████████ (27)
hits=3,████████████████████████████████████████ (33)
hits=4,█████████████████████████ (21)
hits=5,██████ (5)


Hits,Distribution
hits=0,███ (5)
hits=1,██████████████ (22)
hits=2,█████████████████████████ (39)
hits=3,████████████████████████████████████████ (61)
hits=4,██████████████████ (28)
hits=5,█ (2)


Hits,Distribution
hits=0,██ (5)
hits=1,██████████████ (29)
hits=2,██████████████████████ (45)
hits=3,████████████████████████████████████████ (80)
hits=4,███████████████████ (39)
hits=5,█ (2)
